# Calling models on Zenith

In this section we will cover the basics of using the OpenAI API to interact with models on splinter, including:
- Chat completions
- Streaming
- Vision input
- Image generation and editing

The beauty of the OpenAI API is that is very simple to use.

In your environment you should have a file called `.env` with the following:

```bash
HPC_API_KEY="sk-proj-1234567890"
```

We will give you this key in the workshop. __The key will be deactivated after the workshop!__

You can then grab the key using python:


In [ ]:
from openai import OpenAI
import dotenv
import os
from rich.console import Console

# We have this to make our printing very pretty
console = Console(width=88)

dotenv.load_dotenv()

client = OpenAI(
    base_url='https://llm.hpc.cam.ac.uk/v1/',
    api_key=os.getenv("HPC_API_KEY")
)

## Available models

Let's first check and see which models we have access to:

In [ ]:
for model in client.models.list():
    print(model)

## Chat Completions

We'll start with the generic chat completions endpoint. Only some models can be used for chat:

In [ ]:
system_prompt = "You are Matsuo Basho, the great Japanese haiku poet. When asked to provide a haiku, provide only the haiku."
user_query = "Can you give me a haiku about a Samurai cat."

In [ ]:
response = client.chat.completions.create(
  model="zai-org/GLM-5.3",
  reasoning_effort="high",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_query},
  ],
)
print(response.choices[0].message.content)

Purrfect.

We can have a look at the response object as well and see what other information we can see.

In [ ]:
console.print(response)

We can see useful information such as
- The reasoning trace. In this example, it is very short. This is good to know, because a common complaint for some of these open source LLMs is the verbosity of their reasoning traces, even when presented with seemingly trivial tasks.  
- The timestamp.
- Tokens sent and received.
- Function / tool calls.

At Zenith, we do not record any of the content sent to the gateway, nor do we record anything the model produces. We only record the timestamp and token counts, for SLO purposes.

## Other endpoints

This API offers a number of _endpoints_ that allow you to interact with the models. We will address them as they occur, but the first is the `/chat/completions` endpoint. This endpoint allows you to interact with the model in a conversational manner.

Only 2 arguments are actually required for this endpoint:

- `model: str` The model to use. E.g.:
    - `'moonshotai/Kimi-K3'`

    - `'Qwen/Qwen3.8-27B-FP8'`

- `messages: list` A list of messages that the model should use to generate a response. Each entry in the list of messages comes in the form:

```python
{"role": "<role>", "content": "<content>", "name": "<name>"}
```

Where `<role>` can take one of the following forms:

- `'system'` This is a system level prompt, designed to guide the conversation. For example: 

_"You are a customer service bot."_

- `'user'` This is direct input from the user. For example: 

_"How do I reset my password?"_

- `'assistant'` This is the response from the model. For example:

_"To reset your password, please visit our website and click on the 'Forgot Password' link."_

So all of this fed into one message list would look like this:

```python
messages = [
    {"role": "system", "content": "You are a customer service bot."},
    {"role": "user", "content": "How do I reset my password?"},
    {"role": "assistant", "content": "To reset your password, please visit our website and click on the 'Forgot Password' link."}
]
```

A basic `/chat/completions` request could then use one of the available conversational models:

```python
response = client.chat.completions.create(
    model="Qwen/Qwen3.8-27B-FP8",
    messages=messages,
)
```

### Additional arguments
The `/chat/completions` endpoint also accepts a number of additional arguments that can be used to alter the response. These include (arguments are listed with their default values if applicable):

- `max_tokens: int` The maximum number of tokens to generate in the response. Important to stop the model from generating too much text and racking up a huge bill.

- `n: int = 1` The number of completions to generate. This is useful when you want to generate multiple completions and select the best one. You'll be charged for the _**total**_ number of tokens generated across all completions, so be careful with setting this too high.

- `temperature: float = 1.0` The temperature of the model, ranging from 0.0 to 2. Use low values for deterministic responses, and high values for more creative responses.

- `top_p: float = 1.0` The probability of sampling from the top `p` tokens. This is useful for controlling the diversity of the responses. Setting this to a higher values means the model is more likely to sample from a wider range of tokens.

- `logprobs: bool = False` Whether to return the log probabilities of the tokens generated. This is useful when you want to understand how the model is making decisions.

- `logit_bias: dict` A dictionary of logit biases to apply to the tokens. This is useful when you want to guide the model towards generating certain types of responses.

- `response_format: str` The format of the response. We will cover this later...

- `stream: bool = False` Whether to stream the response back to the client. This is useful when you want to get the response in real-time. Nobody likes to sit and wait for a response. Seeing the text generated as and when it is ready is a much better user experience.

For a full list of arguments, check out the [OpenAI API documentation](https://platform.openai.com/docs/api-reference/chat/create).

As of writing `Kimi-K3` and `GLM-5.3` are the current best offering. But we'll stick with `Qwen3.8-27B-FP8` with `"enable_thinking": False` because it is cheaper and still highly capable.
Note that some additional arguments are required. If you don't set `"enable_thinking": False`, the model will generate too much reasoning output...

> NOTE This will not work on Kimi K3 - is no way to turn off thinking.

In [ ]:
response = client.chat.completions.create(
  model="Qwen/Qwen3.8-27B-FP8",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_query},
  ],
  max_completion_tokens=512,
  temperature=1.0,
  top_p=0.95,
  presence_penalty=1.5,
  # reasoning_effort="low",
  extra_body={
      "top_k": 20,
      "chat_template_kwargs": {"enable_thinking": False},
  }, 
)

print(response.choices[0].message.content)

## Streaming a response

Streaming a response is mainly for user experience. It allows the user to see the response as it comes in, rather than waiting for the whole response to come in. For many applications, this might not be necessary.

Let's stream a response using the same model

In [ ]:
stream = client.chat.completions.create(
    # model="zai-org/GLM-5.3",
      model="Qwen/Qwen3.8-27B-FP8",
      reasoning_effort="low",
      messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query},
      ],
    stream=True,
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content is not None:
        print(content, end="", flush=True)

All this really does is create a streaming object, which acts like a generator. We can then print the chunk as it comes in.

## Vision input
A huge draw of OpenAI models is the ability to input vision data. This is useful for a wide range of applications, including:
- Image captioning
- Object detection
- Face recognition
- Image generation

Let's try an example of inputting an image. First we need to look at the image:

![plot](./data/imgs/figure.jpeg)

Here is the caption from this figure:

<table><tr><td>

**Fig. 2 Spatial and temporal self-similarity and correlation in switching activity.**

_(A) Percolating devices produce complex patterns of switching events that are self-similar in nature. The top panel contains 2400 s of data, with the bottom panels showing segments of the data with 10, 100, and 1000 times greater temporal magnification and with 3, 9, and 27 times greater magnification on the vertical scale (units of G0 = 2e2/h, the quantum of conductance, are used for convenience). The activity patterns appear qualitatively similar on multiple different time scales. (B and E) The probability density function (PDF) for changes in total network conductance, P(ΔG), resulting from switching activity exhibits heavy-tailed probability distributions. (C and F) IEIs follow power law distributions, suggestive of correlations between events. (D and G) Further evidence of temporal correlation between events is given by the autocorrelation function (ACF) of the switching activity (red), which decays as a power law over several decades. When the IEI sequence is shuffled (gray), the correlations between events are destroyed, resulting in a significant increase in slope in the ACF. The data shown in (B) to (D) (sample I) were obtained with our standard (slow) sampling rate, and the data shown in (E) to (G) (sample II) were measured 1000 times faster (see Materials and Methods), providing further evidence for self-similarity._
</td></tr></table>

This figure is taken from _[Avalanches and criticality in self-organized nanoscale networks, Mallinson et al., 2019.](https://www.science.org/doi/10.1126/sciadv.aaw8438)_

Now let's use the OpenAI vision model to generate a caption for this figure.

In [ ]:
prompt = (
    "This figure is a caption from a paper entitled Avalanches and criticality in self-organized nanoscale networks. "
    "Please provide a caption for this figure. "
    "You should describe the figure, grouping the panels where appropriate. "
    "Feel free to make any inferences you need to."

)

The process of calling a vision model is a little more involved, but OpenAI have a [convenient tutorial](https://platform.openai.com/docs/guides/vision) on how to do this.

Essentially we need to first convert the image to a base64 string. We can then pass this to the OpenAI API.

In [ ]:
import base64

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


# Path to your image
image_path = "data/imgs/figure.jpeg"

# Getting the Base64 string
base64_image = encode_image(image_path)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
            },
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
]

response = client.chat.completions.create(
    model="Qwen/Qwen3.8-27B-FP8",
    messages=messages,
    temperature=1.0,
    top_p=0.95,
    presence_penalty=1.5,
    extra_body={
      "top_k": 20,
      "chat_template_kwargs": {"enable_thinking": False},
    }, 
)

In [ ]:
print(response.choices[0].message.content)

I mean, I don't know about you, but I think that's incredible. Let's consider what it has done:
- Correctly grouped the panels in the same way the real caption did.
- Provided information on the observation periods.
- Drawn out the important information, such as critical exponents.
- Made the link between power law distributions and scale-free behaviour.

However, it has failed to provide information on temporal correlations, and it has not noticed the self-similarity in caption 1.

But this is still quite impressive, and with more information we could potentially get some better captions.

## OCR
We have a number of models capable of OCR on Zenith. Let's look at both a dedicated OCR model, and a "regular" LLM.

Here is some handwritten text:

<img src="./data/imgs/swp.jpeg" width="40%">

Let's get the model to convert this text to markdown.

In [ ]:
prompt = "Here is some handwritten text and equations in an image. Please read it and convert to markdown, rendering the equations in LaTeX format."

In [ ]:
import base64
import mimetypes


def ask_image(client, model, image_path, prompt, **kwargs):
    """Send an image + prompt to any OpenAI-compatible chat model.

    Extra kwargs are passed through to client.chat.completions.create()
    (e.g. temperature, max_tokens, stream, ...).
    """

    mime = mimetypes.guess_type(image_path)[0] or "image/jpeg"

    with open(image_path, "rb") as f:
        base64_image = base64.b64encode(f.read()).decode("utf-8")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:{mime};base64,{base64_image}"},
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]

    return client.chat.completions.create(model=model, messages=messages, **kwargs)


response = ask_image(
    client,
    model="zai-org/GLM-OCR",
    image_path="./data/imgs/swp.jpeg",
    prompt=prompt,
    reasoning_effort="high",
    temperature=0.0,
)

In [ ]:
from IPython.display import Markdown, display

display(Markdown(response.choices[0].message.content))

Looks good to me. But let's try a bigger model:

In [ ]:
response = ask_image(
    client,
    model="moonshotai/Kimi-K3",
    image_path="./data/imgs/swp.jpeg",
    prompt=prompt,
    reasoning_effort="low",
)

display(Markdown(response.choices[0].message.content))

That took much longer, and I got basically the same result. What about something more complex?

<img src="./data/imgs/curie.jpg" width="40%">

In [ ]:
prompt = "Here is some handwritten text, in French. Please read it and convert to markdown in English, rendering the equations in LaTeX format."

In [ ]:
response = ask_image(
    client,
    model="zai-org/GLM-OCR",
    image_path="./data/imgs/curie.jpg",
    prompt=prompt,
    reasoning_effort="high",
    temperature=0.0
)

display(Markdown(response.choices[0].message.content))

and now with Kimi:

In [ ]:
response = ask_image(
    client,
    model="moonshotai/Kimi-K3",
    image_path="./data/imgs/curie.jpg",
    prompt=prompt,
    reasoning_effort="high",
)

display(Markdown(response.choices[0].message.content))

## Image generation and editing
Sticking with the image theme, let's generate and modify some images

In [ ]:
import base64

In [ ]:
prompt = "Generate an image of a raccoon standing in libary, wearing a suit. He is standing behind a desk. In the style of a 19th century oil painting."

response = client.images.generate(
    model="Tongyi-MAI/Z-Image-Turbo",
    prompt=prompt,
    n=1,
    size="1024x1024",
    response_format="b64_json",
)

In [ ]:
image_data = base64.b64decode(response.data[0].b64_json)
with open("./data/imgs/raccoon.jpeg", "wb") as f:
    f.write(image_data)

<img src="./data/imgs/raccoon.jpeg" width="40%">

Let's edit the image...

In [ ]:
response = client.images.edit(
    model="Qwen/Qwen-Image-Edit-2511",
    image=open("./data/imgs/raccoon.jpeg", "rb"),
    prompt="Give this raccoon a pair of small glasses.",
)

In [ ]:
image_data = base64.b64decode(response.data[0].b64_json)
with open("./data/imgs/raccoon-edited.jpeg", "wb") as f:
    f.write(image_data)

<img src="./data/imgs/raccoon-edited.jpeg?v=2" width="40%">

## Embeddings

Embeddings can be useful for generating high dimensional vectors from text

In [ ]:
response = client.embeddings.create(
    model="Qwen/Qwen3-Embedding-4B",
    input="hello world"
)

print(f"Dimensions: {len(response.data[0].embedding)}")
print(f"First 5 values: {response.data[0].embedding[:5]}")

Let's try something else. We can grab a subset of the imdb dataset and make some embeddings, perhaps for a further classification task later on.

In [ ]:
! pip install datasets

In [ ]:
import os
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")
train = ds["train"]

neg = train.filter(lambda x: x["label"] == 0).select(range(1000))  # first 100 negative
pos = train.filter(lambda x: x["label"] == 1).select(range(1000))  # first 100 positive

for split_name, subset in [("neg", neg), ("pos", pos)]:
    out_dir = os.path.join("data/imdb", split_name)
    os.makedirs(out_dir, exist_ok=True)
    for i, example in enumerate(subset):
        out_path = os.path.join(out_dir, f"{i:03d}.txt")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(example["text"])


In [ ]:
import time
import numpy as np
from tqdm.auto import tqdm

def embed_texts(texts, model="Qwen/Qwen3-Embedding-4B", batch_size=100, max_retries=5):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        for attempt in range(max_retries):
            try:
                response = client.embeddings.create(model=model, input=batch)
                all_embeddings.extend(item.embedding for item in response.data)
                break
            except Exception as e:
                wait = 2 ** attempt 
                print(f"Batch {i//batch_size} failed ({e}) — retrying in {wait}s")
                time.sleep(wait)
        else:
            raise RuntimeError(f"Batch at index {i} failed after {max_retries} retries")
    return all_embeddings


In [ ]:
import glob

paths = sorted(glob.glob("data/imdb/*/*.txt"))
texts = [open(p, encoding="utf-8").read() for p in paths]
labels = [0 if "/neg/" in p else 1 for p in paths]

In [ ]:
len(texts), len(labels)  # should be 2000, 2000

In [ ]:
from datasets import load_dataset

embeddings = embed_texts(texts)

np.savez("data/imdb/imdb_train_embeddings.npz",
         embeddings=np.array(embeddings),
         labels=np.array(labels))

And maybe we can do a PCA and plot them...

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

data = np.load("data/imdb/imdb_train_embeddings.npz")
X = data["embeddings"]
y = data["labels"]  # 0 = negative, 1 = positive

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

plt.figure(figsize=(6, 6))
plt.scatter(X_2d[y == 0, 0], X_2d[y == 0, 1], label="Negative", alpha=0.3, s=5)
plt.scatter(X_2d[y == 1, 0], X_2d[y == 1, 1], label="Positive", alpha=0.3, s=5)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.title("IMDb Review Embeddings — PCA")
plt.legend(markerscale=3)
plt.tight_layout()
plt.show()

## Tools
We can also give the model some tools to use - these are essentially just functions that we can call, and the role of the LLM is to generate the arguments to that function.

Here is a simple example to do some maths.

In [ ]:
system_prompt = "You are a helpful mathematician. You will only solve the problems given to you. Do not provide any additional information. Provide only the answer. If needed, give 5 decimal places of precision. Never give a fraction."
user_query = "What is 1056514 / 1316394?"

response = client.chat.completions.create(
  model="Qwen/Qwen3.8-27B-FP8",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_query},
  ],
)

print(response.choices[0].message.content)

In [ ]:
1056514 / 1316394

In [ ]:
console.print(response.choices[0].message.reasoning_content)

The LLM may even be correct! (When we were running this workshop ~1 year ago, it was not)

To endow the model with "tool use", we add a function:

In [ ]:
def divide(a: float, b: float) -> float:
    return a / b

And then provide the model with a _schema_, which is just a description of the function in dictionary form (or JSON):

In [ ]:
tool_schema = {
    "type": "function",
    "function": {
        "name": "divide",
        "description": "Given two floats, a and b, return a divided by b.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "The divident."
                },
                "b": {
                    "type": "number",
                    "description": "The divisor."
                }
            },
            "required": ["a", "b"],
            "additionalProperties": False,
        }
    }
}

tools = [
    tool_schema
]

When we make function calls with an LLM, we have to let it know that it has access to one or more tools. We do this by passing in the `tools` argument.

In [ ]:
response = client.chat.completions.create(
    model="Qwen/Qwen3.8-27B-FP8",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query},
    ],
    temperature=1.0,
    top_p=0.95,
    presence_penalty=1.5,
    extra_body={
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": False},
    }, 
    tools=tools,
)

In [ ]:
print(response.choices[0].message.tool_calls[0])

So now in our `response`, we have this extra part called `tool_calls` that we can extract information from - in this case the arguments to the `divide` function.

Note that you could achieve a similar result with appropriate prompting - e.g. "Extract only the arguments to a function that multiplies two numbers."

We unpack the actual arguments as a dictionary:

In [ ]:
import json

tool_call = response.choices[0].message.tool_calls[0]
arguments = json.loads(tool_call.function.arguments)
print(arguments)

And we now feed the arguments into our `divide` function:

In [ ]:
result = divide(**arguments)
print(result)

So now we have the answer. We can either just return this, or we can feed it back into the LLM. We need to provide the model with the `tool_calls[0].id`, so that it can associate response messages of the tool type with the correct tool call.

In [ ]:
tool_call = response.choices[0].message.tool_calls[0]

tool_call_id = tool_call.id
tool_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

In [ ]:
tool_response = response

tool_call = tool_response.choices[0].message.tool_calls[0]

tool_call_result = {
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps({
        "a": arguments["a"],
        "b": arguments["b"],
        "result": result,
    }),
}

final_response = client.chat.completions.create(
    model="Qwen/Qwen3.8-27B-FP8",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query},
        tool_response.choices[0].message,
        tool_call_result,
    ],
    max_tokens=100,
    extra_body={
        "top_k": 20,
        "chat_template_kwargs": {
            "enable_thinking": False
        },
    },
)

final_response.choices[0].message.content

This is quite a lot of work to divide two numbers, but of course the power comes when doing more complex tasks.

And this brings to light an interesting contrast. People talk a lot about "agents" and "tools" and "systems", and when we interact with ChatGPT, we get a single coherent experience. Sometimes it is difficult to distinguish between what the LLM is doing, and what the software engineers have built around it in order to create this seemless experience.

### Model-specific parameters

Different models support different optional parameters in addition to the standard chat completion arguments.

- `Qwen/Qwen3.8-27B-FP8`
  - `temperature`
  - `top_p`
  - `max_tokens`
  - `enable_thinking`
  - When served through vLLM, `enable_thinking` may need to be passed through `chat_template_kwargs`.

- `Qwen/Qwen3.6-35B`
  - `temperature`
  - `top_p`
  - `max_tokens`
  - `enable_thinking`
  - When served through vLLM, `enable_thinking` may need to be passed through `chat_template_kwargs`.

- `moonshotai/Kimi-K3`
  - `temperature`
  - `max_tokens`
  - `reasoning_effort`
    - `"low"`
    - `"high"`
    - `"max"`
  - `tools`
  - `tool_choice`
  - Returns reasoning separately as `reasoning_content`.

- `zai-org/GLM-5.3`
  - `temperature`
  - `max_tokens`
  - `reasoning_effort`
    - `"low"`
    - `"high"`
    - `"max"`
  - `clear_thinking`

- `zai-org/GLM-OCR`
  - OCR-specific image/document input parameters.
  - This is not intended to be used as a normal conversational model.

- `Qwen/Qwen3-Embedding-4B`
  - Embedding dimension
  - Input text
  - Task/instruction text
  - Does not support reasoning parameters.

- `openai/Tongyi-MAI/Z-Image-Turbo`
  - Image size / resolution
  - Number of inference steps
  - Guidance scale
  - Other image-generation parameters

- `openai/Qwen/Qwen-Image-Edit-2511`
  - `image`
  - `prompt`
  - `negative_prompt`
  - `true_cfg_scale`
  - `guidance_scale`
  - `num_inference_steps`
  - `num_images_per_prompt`
  - Seed / generator settings

The exact parameters accepted by an API request depend on both the underlying model and how that model is configured in LiteLLM/vLLM.